# 04 — Process Window & Bossung Analysis

This notebook explores the **process window** — the range of dose and focus settings that produce an acceptable CD.

**Key concepts:**
- **Bossung plot**: CD vs. dose at different focus settings
- **Depth of Focus (DoF)**: Focus range where CD stays within tolerance
- **Exposure Latitude (EL)**: Dose range where CD stays within tolerance
- **MEEF**: Mask Error Enhancement Factor

## 1. Setup


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from euv.pipeline import SimulationConfig, run_simulation
from euv.metro.process_window import dose_matrix, process_window, pw_metrics, plot_bossung


## 2. Pipeline Wrapper for the Dose-Focus Sweep

`dose_matrix()` expects a callable `fn(dose_mj_cm2, focus_nm)` returning a dict with `cd_nm`.


In [ ]:
TARGET_CD = 32.0

def pipeline_fn(dose_mj_cm2: float, focus_nm: float) -> dict:
    cfg = SimulationConfig(
        period_nm=64.0,
        line_width_nm=TARGET_CD,
        dose_mj_cm2=dose_mj_cm2,
        focus_nm=focus_nm,
        grid=256,
        se_blur_nm=5.0,  # CAR resist
        resist_model="aerial_threshold",
    )
    result = run_simulation(cfg)
    # CD = 0 means unmeasurable; use NaN for dose_matrix but ensure we don't get all-NaN slices
    cd = result.cd_nm if result.cd_nm > 0 else np.nan
    return {"cd_nm": cd, "nils_value": result.nils_value}


## 3. Compute the Bossung Matrix (Dose × Focus)

This runs 16 × 21 = 336 simulations — takes a couple of minutes on CPU.


In [ ]:
doses = np.linspace(15, 35, 11)       # mJ/cm² — bracket dose-to-size
focuses = np.linspace(-60, 60, 13)    # nm — physical DoF range
tolerance = 0.10  # ±10%

print(f"Computing {len(focuses)}×{len(doses)} Bossung matrix...")
cd_mat = dose_matrix(pipeline_fn, doses.tolist(), focuses.tolist(), TARGET_CD, tolerance)

nils_mat = np.zeros_like(cd_mat)
for i, f in enumerate(focuses):
    for j, d in enumerate(doses):
        nils_mat[i, j] = pipeline_fn(d, f)["nils_value"]

print(f"CD range: {np.nanmin(cd_mat):.2f} – {np.nanmax(cd_mat):.2f} nm")
print(f"NILS range: {np.nanmin(nils_mat):.3f} – {np.nanmax(nils_mat):.3f}")


## 4. Extract Process Window Metrics


In [ ]:
pw = process_window(cd_mat, doses.tolist(), focuses.tolist(), TARGET_CD, tolerance)

print("=== Process Window Metrics ===")
print(f"Target CD:         {TARGET_CD:.1f} nm")
print(f"Tolerance:         ±{tolerance*100:.0f}% (±{TARGET_CD*tolerance:.1f} nm)")
print(f"Best dose:         {pw['best_dose']:.1f} mJ/cm²")
print(f"Best focus:        {pw['best_focus']:.1f} nm")
print(f"Depth of Focus:    {pw['dof_nm']:.1f} nm")
print(f"Exposure Latitude: {pw['el_pct']:.1f}%")

metrics = pw_metrics(cd_mat, TARGET_CD, tolerance, nils_matrix=nils_mat)
print(f"\nMax NILS (in spec): {metrics['max_nils']:.3f}")
print(f"Min NILS (in spec): {metrics['min_nils']:.3f}")
print(f"In-spec points:     {metrics['n_in_spec']} / {cd_mat.size}")


## 5. Bossung ASCII Table


In [ ]:
plot_bossung(cd_mat, doses.tolist(), focuses.tolist())


## 6. Heatmap Visualisation (CD, NILS, In-Spec Mask)


In [ ]:
cd_low = TARGET_CD * (1 - tolerance)
cd_high = TARGET_CD * (1 + tolerance)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im1 = axes[0].imshow(
    cd_mat.T, origin="lower", aspect="auto",
    extent=[focuses[0], focuses[-1], doses[0], doses[-1]],
    cmap="RdYlGn", vmin=cd_low, vmax=cd_high,
)
axes[0].set_xlabel("Focus [nm]")
axes[0].set_ylabel("Dose [mJ/cm²]")
axes[0].set_title(f"CD Heatmap (target={TARGET_CD:.0f} nm, ±{tolerance*100:.0f}%)")
axes[0].contour(focuses, doses, cd_mat.T, levels=[cd_low, cd_high],
                colors="k", linewidths=1, linestyles="--")
plt.colorbar(im1, ax=axes[0], label="CD [nm]")

im2 = axes[1].imshow(
    nils_mat.T, origin="lower", aspect="auto",
    extent=[focuses[0], focuses[-1], doses[0], doses[-1]],
    cmap="viridis",
)
axes[1].set_xlabel("Focus [nm]")
axes[1].set_ylabel("Dose [mJ/cm²]")
axes[1].set_title("NILS Heatmap")
plt.colorbar(im2, ax=axes[1], label="NILS")

in_spec = (cd_mat >= cd_low) & (cd_mat <= cd_high) & (~np.isnan(cd_mat))
im3 = axes[2].imshow(
    in_spec.T, origin="lower", aspect="auto",
    extent=[focuses[0], focuses[-1], doses[0], doses[-1]],
    cmap="Greys", vmin=0, vmax=1,
)
axes[2].set_xlabel("Focus [nm]")
axes[2].set_ylabel("Dose [mJ/cm²]")
axes[2].set_title(f"In-Spec Region (DoF={pw['dof_nm']:.0f} nm, EL={pw['el_pct']:.1f}%)")
plt.colorbar(im3, ax=axes[2], label="In spec")

plt.tight_layout()
plt.show()


## 7. Bossung Curves (CD vs Dose at Several Focus Settings)


In [ ]:
focus_indices = [0, len(focuses)//4, len(focuses)//2, 3*len(focuses)//4, -1]

plt.figure(figsize=(10, 6))
for idx in focus_indices:
    plt.plot(doses, cd_mat[idx, :], "o-", label=f"Focus {focuses[idx]:+.0f} nm")

plt.axhline(TARGET_CD, color="k", linestyle=":", label=f"Target CD = {TARGET_CD:.0f} nm")
plt.axhspan(cd_low, cd_high, alpha=0.15, color="green", label=f"±{tolerance*100:.0f}% spec")
plt.xlabel("Dose [mJ/cm²]")
plt.ylabel("CD [nm]")
plt.title("Bossung Curves: CD vs Dose at Various Focus Settings")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 8. Focus Dependence at Best Dose


In [ ]:
best_focus_row = int(np.argmin(np.abs(focuses - pw["best_focus"])))
best_dose_idx = int(np.nanargmin(np.abs(cd_mat[best_focus_row, :] - TARGET_CD)))
best_dose_val = doses[best_dose_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(focuses, cd_mat[:, best_dose_idx], "o-", color="blue")
ax1.axhline(TARGET_CD, color="k", linestyle=":", label=f"Target = {TARGET_CD:.0f} nm")
ax1.axhspan(cd_low, cd_high, alpha=0.15, color="green", label=f"±{tolerance*100:.0f}% spec")
ax1.axvline(pw["best_focus"], color="red", linestyle="--", alpha=0.7,
            label=f"Best focus = {pw['best_focus']:.0f} nm")
ax1.set_xlabel("Focus [nm]")
ax1.set_ylabel("CD [nm]")
ax1.set_title(f"CD vs Focus at Dose = {best_dose_val:.1f} mJ/cm² (DoF = {pw['dof_nm']:.1f} nm)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(focuses, nils_mat[:, best_dose_idx], "s-", color="orange")
ax2.axvline(pw["best_focus"], color="red", linestyle="--", alpha=0.7)
ax2.set_xlabel("Focus [nm]")
ax2.set_ylabel("NILS")
ax2.set_title(f"NILS vs Focus at Dose = {best_dose_val:.1f} mJ/cm²")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9. SE Blur Impact on the Process Window

SE blur is the dominant resolution limiter in EUV — here its effect on DoF and EL.


In [ ]:
se_blurs = np.linspace(0, 15, 8)
dofs, els, pw_areas = [], [], []

for blur in se_blurs:
    def pipeline_fn_blur(dose, focus):
        cfg = SimulationConfig(
            period_nm=64.0, line_width_nm=TARGET_CD,
            dose_mj_cm2=dose, focus_nm=focus,
            grid=256, se_blur_nm=blur, resist_model="aerial_threshold",
        )
        r = run_simulation(cfg)
        cd = r.cd_nm if r.cd_nm > 0 else np.nan
        return {"cd_nm": cd}

    cd_b = dose_matrix(pipeline_fn_blur, doses.tolist(), focuses.tolist(), TARGET_CD, tolerance)
    # If the whole matrix is NaN (unmeasurable), skip process_window metrics
    if np.all(np.isnan(cd_b)):
        dofs.append(0.0)
        els.append(0.0)
        pw_areas.append(0.0)
        continue
    pw_b = process_window(cd_b, doses.tolist(), focuses.tolist(), TARGET_CD, tolerance)
    dofs.append(pw_b["dof_nm"])
    els.append(pw_b["el_pct"])
    pw_areas.append(pw_b["dof_nm"] * pw_b["el_pct"])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(se_blurs, dofs, "o-")
axes[0].set_xlabel("SE Blur σ [nm]"); axes[0].set_ylabel("DoF [nm]")
axes[0].set_title("DoF vs SE Blur"); axes[0].grid(True, alpha=0.3)

axes[1].plot(se_blurs, els, "s-", color="orange")
axes[1].set_xlabel("SE Blur σ [nm]"); axes[1].set_ylabel("EL [%]")
axes[1].set_title("EL vs SE Blur"); axes[1].grid(True, alpha=0.3)

axes[2].plot(se_blurs, pw_areas, "^-", color="green")
axes[2].set_xlabel("SE Blur σ [nm]"); axes[2].set_ylabel("DoF × EL")
axes[2].set_title("Process Window Area vs SE Blur"); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

for b, d, e in zip(se_blurs, dofs, els):
    print(f"blur={b:4.1f} nm: DoF={d:5.1f} nm, EL={e:5.1f}%")


## 10. NA Dependence of the Process Window

Higher NA improves resolution but narrows the depth of focus.


In [ ]:
nas = [0.33, 0.40, 0.55]
dofs_na, els_na = [], []

for na in nas:
    def pipeline_fn_na(dose, focus):
        cfg = SimulationConfig(
            period_nm=64.0, line_width_nm=TARGET_CD,
            dose_mj_cm2=dose, focus_nm=focus,
            grid=256, se_blur_nm=5.0, na=na, resist_model="aerial_threshold",
        )
        r = run_simulation(cfg)
        cd = r.cd_nm if r.cd_nm > 0 else np.nan
        return {"cd_nm": cd}

    cd_na = dose_matrix(pipeline_fn_na, doses.tolist(), focuses.tolist(), TARGET_CD, tolerance)
    if np.all(np.isnan(cd_na)):
        dofs_na.append(0.0)
        els_na.append(0.0)
        continue
    pw_na = process_window(cd_na, doses.tolist(), focuses.tolist(), TARGET_CD, tolerance)
    dofs_na.append(pw_na["dof_nm"])
    els_na.append(pw_na["el_pct"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(nas, dofs_na, "o-")
ax1.set_xlabel("NA"); ax1.set_ylabel("DoF [nm]")
ax1.set_title("DoF vs NA"); ax1.grid(True, alpha=0.3)

ax2.plot(nas, els_na, "s-", color="orange")
ax2.set_xlabel("NA"); ax2.set_ylabel("EL [%]")
ax2.set_title("EL vs NA"); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

for na, d, e in zip(nas, dofs_na, els_na):
    print(f"NA={na:.2f}: DoF={d:.1f} nm, EL={e:.1f}%")


## 11. MEEF — Mask Error Enhancement Factor

$$MEEF = \frac{\Delta CD_{wafer} / CD_{wafer}}{\Delta CD_{mask} / CD_{mask}}$$

Typical EUV values: 2–5. Computed numerically via two simulations with slightly different mask CD.


In [ ]:
def compute_meef(mask_cd=32.0, delta_mask=1.0, dose=20.0, focus=0.0, se_blur=5.0):
    def run(cd):
        cfg = SimulationConfig(
            period_nm=64.0, line_width_nm=cd,
            dose_mj_cm2=dose, focus_nm=focus,
            grid=256, se_blur_nm=se_blur, resist_model="aerial_threshold",
        )
        return run_simulation(cfg).cd_nm

    cd_nom = run(mask_cd)
    cd_pert = run(mask_cd + delta_mask)
    meef = (abs(cd_pert - cd_nom) / cd_nom) / (delta_mask / mask_cd)
    return meef, cd_nom, cd_pert

meef, cd_nom, cd_pert = compute_meef()
print(f"MEEF: {meef:.3f}")
print(f"Nominal wafer CD:   {cd_nom:.2f} nm")
print(f"Perturbed wafer CD: {cd_pert:.2f} nm (mask CD +1 nm)")
print(f"Wafer CD change:    {abs(cd_pert - cd_nom):.3f} nm per 1 nm mask CD change")


## 12. Export (CSV + Metrics)

The CLI equivalent: `euv process-window --period=64 --cd=32 --output-plot=pw.png --output-csv=pw.csv`.


In [ ]:
import csv, json

with open("process_window_cd.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([""] + [f"{d:.1f}" for d in doses])
    for j, fc in enumerate(focuses):
        writer.writerow([f"{fc:.0f}"] + [f"{cd_mat[j, i]:.2f}" for i in range(len(doses))])

with open("process_window_metrics.json", "w") as f:
    json.dump({
        "target_cd_nm": TARGET_CD,
        "tolerance_pct": tolerance * 100,
        "dof_nm": pw["dof_nm"],
        "el_pct": pw["el_pct"],
        "best_dose": pw["best_dose"],
        "best_focus": pw["best_focus"],
        "meef": meef,
    }, f, indent=2)

print("✅ Exported process_window_cd.csv and process_window_metrics.json")


## Summary

| Metric | Meaning |
|--------|---------|
| **DoF** | Focus range where CD stays in spec |
| **EL** | Dose range where CD stays in spec |
| **MEEF** | Mask CD error amplification factor |
| **DoF × EL** | Process window area (figure of merit) |

**Key insights:**
- SE blur is the dominant factor shrinking the process window
- Higher NA improves resolution but narrows DoF
- MEEF > 1 means mask errors are magnified on wafer

**Next steps:**
- `euv process-window --period=64 --cd=32 --output-plot=pw.png --output-csv=pw.csv`
- Calibrate resist parameters against experimental Bossung data (`euv calibrate`)
- Explore illumination shapes (dipole, quasar) for process-window optimisation
